In [1]:
# REQUIRED inputs you set: # fn1
pl_name = "https://soundcloud.com/yeriko_dj/sets/denver?si=cea5a446b9cf44b4a76bc09e1b3a2591&utm_source=clipboard&utm_medium=text&utm_campaign=social_sharing"
#"https://soundcloud.com/yeriko_dj/sets/pocket-change-wknd/s-hEjR9L4yKsw?si=900df7eeffb94fa3bad3bc285eb6659b&utm_source=clipboard&utm_medium=text&utm_campaign=social_sharing"

#"https://soundcloud.com/yeriko_dj/sets/new-house?si=0373ee6a3b774170b06fefc4e85ff38f&utm_source=clipboard&utm_medium=text&utm_campaign=social_sharing"

#"https://soundcloud.com/yeriko_dj/sets/new-latin-sexy-dance?si=1b81c18d4cef4b17a4c0a8e861fbaea4&utm_source=clipboard&utm_medium=text&utm_campaign=social_sharing"

#"https://soundcloud.com/yeriko_dj/sets/ragaeton?si=3590778e849c48b59d12f6373f11bf6c&utm_source=clipboard&utm_medium=text&utm_campaign=social_sharing"
#"https://soundcloud.com/yeriko_dj/sets/detroit/s-8WfZtiBirPg?si=3721e0d9f156428a8a78d0603049fb1c&utm_source=clipboard&utm_medium=text&utm_campaign=social_sharing"
#"https://soundcloud.com/yeriko_dj/sets/idk/s-97gP3fJbrkh?si=29121c29b03c4d88bbadd2f89de16c8b&utm_source=clipboard&utm_medium=text&utm_campaign=social_sharing"
#"https://soundcloud.com/yeriko_dj/sets/salsa_aug18/s-4lRpgd7Ys0H?si=1a974f75116a41b68fb98def6fa33452&utm_source=clipboard&utm_medium=text&utm_campaign=social_sharing"
#pl_name =  "https://soundcloud.com/user862007976/sets/salsa?si=13f9da5ee0074af1aa9b6e7b919f6e94&utm_source=clipboard&utm_medium=text&utm_campaign=social_sharing"
# REQUIRED inputs you set: fn2
OUT_DIR = "/Users/yerik/Downloads/_soundcloud_audio"  # your target folder (will be created)
src_folder =OUT_DIR
EXT     = "mp3"                                       # "mp3" (320), "wav", or "aiff"
KBPS    = 320                                         # only for mp3
COOKIES = None                                        # e.g., "/Users/yerik/Downloads/cookies.txt" if needed

# RUN

In [3]:
# Get urls from playlists 
df_tracks = _soundcloud_1808_pl2name_GET_df(pl_name, genre="Salsa", purchase_date="08-18-2025")
#df_tracks = _soundcloud_1808_pl2name_GET_df(pl_name)
df_tracks.head(2)

                            # e.g., "/Users/yerik/Downloads/cookies.txt" if needed

# df_tracks must exist and contain 'src_playlist' with SoundCloud URLs
df_tracks = _sc_1808_df_GET_audio_for_src_playlist_APPEND(
    df_tracks,
    out_dir=OUT_DIR,
    ext=EXT,
    prefer_bitrate=KBPS,
    cookies_path=COOKIES
)

# (Optional) Save the augmented DF
# df_tracks.to_csv("/Users/yerik/Downloads/_soundcloud_audio/_dl_log.csv", index=False)


TQM • Downloading SoundCloud audio:   0%|                                             | 0/10 [00:00<?, ?url/s]

TQM • Downloading SoundCloud audio: 100%|████████████████████████████████████| 10/10 [00:57<00:00,  5.78s/url]


# FUNCTIONS 

#### # Get urls from playlists 

In [2]:
import re, json, time
import requests
import pandas as pd
from datetime import datetime
from tqdm import tqdm

def _soundcloud_1808_pl2name_GET_df(playlist_links, genre="", purchase_date=""):
    """
    Input:
      - playlist_links: str | list/Series[Index] of SoundCloud playlist URLs
      - genre: optional string; if empty, prompts once for the batch
      - purchase_date: optional 'YYYY-MM-DD' (blank = keep empty purchase fields)
    Output:
      - pandas.DataFrame with columns:
        ['src_playlist','track_title','track_artist','track_url','track_id','error',
         'genre','mix_name','remixers','label','key','bpm',
         'release_year','release_month','release_day',
         'purchase_year','purchase_month','purchase_day','new_name']
    """
    def _as_list(x):
        if isinstance(x, (list, tuple, pd.Series, pd.Index)): return list(x)
        return [x]

    def _request(url):
        return requests.get(url, headers={
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.6 Safari/605.1.15",
            "Accept": "text/html,application/json;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
        }, timeout=25)

    def _scrape_tracks(pl_url):
        html = _request(pl_url).text
        m = re.search(r"window\.__sc_hydration\s*=\s*(\[[\s\S]*?\])\s*;", html)
        if not m: 
            return []
        try:
            hydration = json.loads(m.group(1))
        except Exception:
            return []
        pl_objs = [o for o in hydration if isinstance(o, dict) and str(o.get("hydratable","")).startswith("playlist")]
        tracks = []
        for obj in pl_objs:
            data = obj.get("data") or {}
            tlist = data.get("tracks") or (data.get("playlist") or {}).get("tracks") or []
            for t in tlist:
                u = t.get("user") or {}
                tracks.append({
                    "src_playlist": pl_url,
                    "track_title": t.get("title",""),
                    "track_artist": u.get("username","") or u.get("permalink",""),
                    "track_url": t.get("permalink_url","") or "",
                    "track_id": t.get("id"),
                    "error": ""
                })
        return tracks

    links = _as_list(playlist_links)
    rows = []
    for pl in tqdm(links, desc="TQM • Scraping playlists", unit="playlist"):
        try:
            got = _scrape_tracks(str(pl).strip())
            if not got:
                rows.append({"src_playlist": pl, "track_title":"","track_artist":"","track_url":"","track_id":None,"error":"no_tracks"})
            else:
                rows.extend(got)
        except Exception as e:
            rows.append({"src_playlist": pl, "track_title":"","track_artist":"","track_url":"","track_id":None,"error":str(e)})
        time.sleep(0.8)

    df = pd.DataFrame(rows, columns=["src_playlist","track_title","track_artist","track_url","track_id","error"])
    if df.empty:
        return df

    # --- Batch genre & purchase split ---
    if not genre:
        try:
            genre = input("Enter GENRE for this batch: ").strip()
        except Exception:
            genre = ""
    df["genre"] = genre

    py = pm = pdm = ""
    if purchase_date:
        try:
            pdt = datetime.strptime(purchase_date, "%Y-%m-%d")
            py, pm, pdm = str(pdt.year), f"{pdt.month:02d}", f"{pdt.day:02d}"
        except Exception:
            py = pm = pdm = ""
    df["purchase_year"] = py
    df["purchase_month"] = pm
    df["purchase_day"] = pdm

    # --- Parsing heuristics ---
    for c in ["mix_name","remixers","label","key","bpm","release_year","release_month","release_day"]:
        if c not in df.columns: df[c] = ""

    rx_mix = re.compile(r"\(([^)]*?(?:mix|edit|version|dub|instrumental)[^)]*)\)", re.IGNORECASE)
    rx_remix = re.compile(r"\(([^)]*?remix[^)]*)\)", re.IGNORECASE)
    rx_key_music = re.compile(r"\b([A-G](?:#|b)?\s?(?:maj(?:or)?|min(?:or)?|m|M))\b")
    rx_key_camelot = re.compile(r"\b(1[0-2]|[1-9])[AB]\b", re.IGNORECASE)
    rx_bpm = re.compile(r"\b(\d{2,3})\s?bpm\b", re.IGNORECASE)
    rx_bpm_brackets = re.compile(r"\[(\d{2,3})\]")

    mix_list, remixers_list, key_list, bpm_list = [], [], [], []
    for t in tqdm(df["track_title"].fillna("").astype(str).tolist(), desc="TQM • Parsing titles", unit="track"):
        mix = ""
        m = rx_mix.search(t)
        if m: mix = m.group(1).strip()

        rem = ""
        mr = rx_remix.search(t)
        if mr:
            inner = mr.group(1)
            rem = re.split(r"remix", inner, flags=re.IGNORECASE)[0].strip(" -&x,").strip()

        key_val = ""
        mk = rx_key_music.search(t)
        if mk:
            key_val = mk.group(1).strip().replace("major","maj").replace("minor","min")
        else:
            kc = rx_key_camelot.search(t)
            if kc: key_val = kc.group(0).upper()

        bpm_val = ""
        mb = rx_bpm.search(t) or rx_bpm_brackets.search(t)
        if mb: bpm_val = mb.group(1)

        mix_list.append(mix)
        remixers_list.append(rem)
        key_list.append(key_val)
        bpm_list.append(bpm_val)

    df["mix_name"] = df["mix_name"].where(df["mix_name"].ne(""), mix_list)
    df["remixers"] = df["remixers"].where(df["remixers"].ne(""), remixers_list)
    df["key"] = df["key"].where(df["key"].ne(""), key_list)
    df["bpm"] = df["bpm"].where(df["bpm"].ne(""), bpm_list)

    def nz(x): return "" if pd.isna(x) else str(x)

    df["new_name"] = (
        "TRkw_" + df["track_title"].map(nz) +
        "_ARkw_" + df["track_artist"].map(nz) +
        "_MXkw_" + df["mix_name"].map(nz) +
        "_KYkw_" + df["key"].map(nz) +
        "_BPkw_" + df["bpm"].map(nz) +
        "_GNkw_" + df["genre"].map(nz) +
        "_RMkw_" + df["remixers"].map(nz) +
        "_LBkw_" + df["label"].map(nz) +
        "_RYkw_" + df["release_year"].map(nz) + "_" + df["release_month"].map(nz) + "_" + df["release_day"].map(nz) +
        "_PYkw_" + df["purchase_year"].map(nz) + "_" + df["purchase_month"].map(nz) + "_" + df["purchase_day"].map(nz)
    ).str.replace(r"[\/\\:*?\"<>|]", "_", regex=True)

    return df

# -----######-----###### CORE IMPORTABLE FUNCTION (SoundCloud DF Downloader — DF['src_playlist'] → files) -----######-----###### #
import os, re, json, shutil
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

try:
    import pandas as pd
except Exception:
    pd = None


# --- internal: env checks (no ASCII) ---
def _sc__ensure_ffmpeg():
    from shutil import which
    return which("ffmpeg") is not None

def _sc__safe_name(s):
    # filesystem-safe slug
    s = str(s or "").strip()
    s = re.sub(r"[\\/:*?\"<>|\n\r\t]", "_", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s[:200] if s else "unnamed"

def _sc__build_opts(out_dir, ext, prefer_bitrate, archive_path, cookies_path):
    # postproc: convert to target ext
    postproc = []
    if ext.lower() in ("mp3", "wav", "aiff"):
        postproc = [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": ext.lower(),
            "preferredquality": str(prefer_bitrate) if ext.lower()=="mp3" else "0",
        }]

    # Filename template: Uploader - Title [id].ext inside out_dir
    outtmpl = str(Path(out_dir) / "%(uploader)s - %(title)s [%(id)s].%(ext)s")

    opts = {
        "outtmpl": outtmpl,
        "noplaylist": False,              # if a playlist URL appears, we’ll let yt-dlp handle entries
        "quiet": True,
        "no_warnings": True,
        "ignoreerrors": False,
        "retries": 5,
        "continuedl": True,
        "format": "bestaudio/best",
        "concurrent_fragment_downloads": 4,
        "writethumbnail": False,
        "addmetadata": True,
        "prefer_ffmpeg": True,
        "postprocessors": postproc,
        "restrictfilenames": False,
        "windowsfilenames": False,
        "nooverwrites": True,            # do NOT overwrite existing files
        "download_archive": str(archive_path),  # skip already downloaded ids
    }
    if cookies_path:
        opts["cookiefile"] = str(cookies_path)
    return opts

def _sc__extract_saved_path(info):
    cand = None
    if not info:
        return None
    # Try direct
    if "requested_downloads" in info and info["requested_downloads"]:
        cand = info["requested_downloads"][0].get("filepath")
    # Playlist entry?
    if not cand and "entries" in info and info["entries"]:
        e = info["entries"][0]
        if e and "requested_downloads" in e and e["requested_downloads"]:
            cand = e["requested_downloads"][0].get("filepath")
        elif e and "filepath" in e:
            cand = e["filepath"]
    if not cand and "filepath" in info:
        cand = info["filepath"]
    return Path(cand) if cand else None

def _sc__download_one(url, out_dir, ext, prefer_bitrate, archive_path, cookies_path):
    try:
        import yt_dlp
    except Exception as e:
        raise RuntimeError("yt-dlp not installed. Run: pip install yt-dlp") from e

    if ext.lower() in ("mp3","wav","aiff") and not _sc__ensure_ffmpeg():
        raise RuntimeError("ffmpeg not found. Install with: brew install ffmpeg")

    ydl_opts = _sc__build_opts(out_dir, ext, prefer_bitrate, archive_path, cookies_path)

    # progress hook (quiet by default; reserved for future per-fragment logs)
    def _hook(_d): 
        pass
    ydl_opts["progress_hooks"] = [_hook]

    meta = {
        "source_url": url, "title": None, "uploader": None, "duration_sec": None,
        "id": None, "ext": None, "requested_ext": ext.lower(), "filepath": None,
        "filesize_approx": None, "filesize_bytes": None, "error": None, "status": None
    }

    # Detect “already downloaded” quickly via archive line (best-effort);
    # yt-dlp itself will skip by archive and return fast.
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            # When skipped by archive, info may be None → try to probe id via “simulate”
            if not info:
                try:
                    sim_opts = ydl_opts.copy()
                    sim_opts.update({"skip_download": True})
                    with yt_dlp.YoutubeDL(sim_opts) as ydl_sim:
                        info = ydl_sim.extract_info(url, download=False)
                        meta["status"] = "skipped_archive"
                except Exception:
                    meta["status"] = "skipped_archive"
            else:
                meta["status"] = "downloaded"

        if info:
            meta["title"] = info.get("title")
            meta["uploader"] = info.get("uploader")
            meta["duration_sec"] = info.get("duration")
            meta["id"] = info.get("id")
            meta["ext"] = info.get("ext")
            meta["filesize_approx"] = info.get("filesize_approx")

        saved = _sc__extract_saved_path(info)
        if saved and saved.exists():
            meta["filepath"] = str(saved)
            meta["filesize_bytes"] = saved.stat().st_size
        else:
            # If archive skip, try to resolve the existing path on disk by building the expected pattern:
            if meta.get("id") and meta.get("title") and meta.get("uploader"):
                # Try any ext since postproc could differ
                base_glob = f"{_sc__safe_name(meta['uploader'])} - {_sc__safe_name(meta['title'])} [{meta['id']}]"
                candidates = list(Path(out_dir).glob(base_glob + ".*"))
                if candidates:
                    meta["filepath"] = str(candidates[0])
                    meta["filesize_bytes"] = candidates[0].stat().st_size
        return meta

    except yt_dlp.utils.DownloadError as e:
        meta["error"] = f"DownloadError: {e}"
        meta["status"] = "failed"
        return meta
    except Exception as e:
        meta["error"] = str(e)
        meta["status"] = "failed"
        return meta


# -----######-----###### MAIN IMPORTABLE FUNCTION -----######-----###### #
def _sc_1808_df_GET_audio_for_src_playlist_APPEND(
    df_tracks,
    out_dir,                  # REQUIRED: you provide this absolute path
    ext="mp3",                # "mp3" (320), "wav", "aiff"
    prefer_bitrate=320,       # only for mp3
    cookies_path=None         # optional path to cookies.txt for authenticated access
):
    """
    From df_tracks['src_playlist'] SoundCloud URLs, download audio into out_dir.
    - Never overwrites existing files (yt-dlp nooverwrites + archive).
    - Re-running will only add new tracks (download archive).
    - Appends result columns to df_tracks and returns the augmented DF.

    Columns appended/updated:
      dl_title, dl_uploader, dl_duration_sec, dl_id, dl_ext, dl_requested_ext,
      dl_filepath, dl_filesize_approx, dl_filesize_bytes, dl_error, dl_status

    Notes:
      * For private/unlisted but accessible to you, pass a cookies.txt exported from your browser.
      * Only download content you own or have rights to and respect the site’s TOS.
    """
    if pd is None:
        raise RuntimeError("pandas is required. Run: pip install pandas")

    if "src_playlist" not in df_tracks.columns:
        raise ValueError("DataFrame must contain a 'src_playlist' column with SoundCloud URLs.")

    out_dir = Path(out_dir).expanduser().resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    archive_path = out_dir / "_download_archive.txt"   # tracks already fetched
    archive_path.touch(exist_ok=True)

    # Prepare output columns
    add_cols = [
        "dl_title","dl_uploader","dl_duration_sec","dl_id","dl_ext","dl_requested_ext",
        "dl_filepath","dl_filesize_approx","dl_filesize_bytes","dl_error","dl_status"
    ]
    df_out = df_tracks.copy()
    for c in add_cols:
        if c not in df_out.columns:
            df_out[c] = None

    urls = df_out["src_playlist"].astype(str).fillna("").tolist()

    # TQM BAR
    for i, url in enumerate(tqdm(urls, desc="TQM • Downloading SoundCloud audio", unit="url")):
        url_s = url.strip()
        if not url_s:
            df_out.at[i, "dl_status"] = "skip_empty"
            continue
        md = _sc__download_one(
            url=url_s,
            out_dir=out_dir,
            ext=ext,
            prefer_bitrate=prefer_bitrate,
            archive_path=archive_path,
            cookies_path=(Path(cookies_path).expanduser() if cookies_path else None)
        )

        df_out.at[i, "dl_title"]            = md.get("title")
        df_out.at[i, "dl_uploader"]         = md.get("uploader")
        df_out.at[i, "dl_duration_sec"]     = md.get("duration_sec")
        df_out.at[i, "dl_id"]               = md.get("id")
        df_out.at[i, "dl_ext"]              = md.get("ext")
        df_out.at[i, "dl_requested_ext"]    = md.get("requested_ext")
        df_out.at[i, "dl_filepath"]         = md.get("filepath")
        df_out.at[i, "dl_filesize_approx"]  = md.get("filesize_approx")
        df_out.at[i, "dl_filesize_bytes"]   = md.get("filesize_bytes")
        df_out.at[i, "dl_error"]            = md.get("error")
        df_out.at[i, "dl_status"]           = md.get("status")

    return df_out


##### 2

# 3 normalize and rename 

In [19]:
# # -----######-----###### CORE IMPORTABLE FUNCTION (MP3 → Batch Cover • Strict Rename • Loudnorm→AIFF) -----######-----###### #
# import os, re, io, json, subprocess
# from pathlib import Path
# from datetime import datetime
# import pandas as pd
# from tqdm import tqdm

# # --- Audio/Tags/Art deps
# from mutagen.id3 import (
#     ID3, ID3NoHeaderError, TIT2, TPE1, TPE2, TKEY, TBPM, TCON,
#     TPE4, TPUB, TDRC, TXXX, APIC, COMM
# )
# from mutagen.aiff import AIFF
# from PIL import Image, ImageDraw, ImageFont

# # ---------- helpers ----------
# def _nz(x):
#     if x is None:
#         return ""
#     if isinstance(x, float):
#         try:
#             return str(int(x)) if x.is_integer() else str(x)
#         except Exception:
#             return str(x)
#     return str(x)

# def _safe_name(s):
#     s = _nz(s)
#     return re.sub(r'[\/\\:\*\?"<>\|]', "_", s).strip()

# def _read_id3_fields(p):
#     """
#     Robustly read common fields from MP3; fall back safely.
#     Output keys match your rename spec.
#     """
#     out = {
#         "track_title": "", "track_artist": "", "mix_name": "",
#         "key": "", "bpm": "", "genre": "", "remixers": "",
#         "label": "", "release_year": "", "release_month": "", "release_day": "",
#         "purchase_year": "", "purchase_month": "", "purchase_day": ""
#     }
#     try:
#         audio = ID3(str(p))
#     except ID3NoHeaderError:
#         return out
#     except Exception:
#         return out

#     def _txt_one(frame_name):
#         try:
#             frames = audio.getall(frame_name)
#             if frames and hasattr(frames[0], "text") and frames[0].text:
#                 return _nz(frames[0].text[0])
#         except Exception:
#             pass
#         return ""

#     out["track_title"]  = _txt_one("TIT2")
#     out["track_artist"] = _txt_one("TPE1")
#     # MIX name: common variants live in TXXX with desc "MIXNAME", but fallbacks exist
#     mix_val = ""
#     try:
#         for f in audio.getall("TXXX"):
#             desc = (getattr(f, "desc", "") or "").lower()
#             if desc in ("mixname", "mix", "version"):
#                 if f.text:
#                     mix_val = _nz(f.text[0])
#                     break
#     except Exception:
#         pass
#     if not mix_val:
#         # Weak fallback: album artist or comment sometimes includes mix/version
#         mix_val = _txt_one("TPE2")
#         if not mix_val:
#             try:
#                 comms = audio.getall("COMM")
#                 if comms and hasattr(comms[0], "text") and comms[0].text:
#                     mix_val = _nz(comms[0].text[0])
#             except Exception:
#                 pass
#     out["mix_name"] = mix_val

#     out["key"]      = _txt_one("TKEY")
#     out["bpm"]      = _txt_one("TBPM")
#     out["genre"]    = _txt_one("TCON")
#     out["remixers"] = _txt_one("TPE4")
#     out["label"]    = _txt_one("TPUB")

#     # Release date (TDRC) may be YYYY or YYYY-MM or YYYY-MM-DD
#     try:
#         tdrc = audio.get("TDRC")
#         if tdrc and tdrc.text:
#             ds = _nz(tdrc.text[0])
#             parts = re.split(r"[-:T ]", ds)
#             if len(parts) > 0: out["release_year"]  = parts[0]
#             if len(parts) > 1: out["release_month"] = parts[1]
#             if len(parts) > 2: out["release_day"]   = parts[2]
#     except Exception:
#         pass

#     # Purchase date via TXXX("PurchaseDate"/variants)
#     try:
#         for f in audio.getall("TXXX"):
#             desc = (getattr(f, "desc", "") or "").lower()
#             if desc in ("purchasedate", "purchase_date", "pdate"):
#                 if f.text:
#                     ds = _nz(f.text[0])
#                     parts = re.split(r"[-:T ]", ds)
#                     if len(parts) > 0: out["purchase_year"]  = parts[0]
#                     if len(parts) > 1: out["purchase_month"] = parts[1]
#                     if len(parts) > 2: out["purchase_day"]   = parts[2]
#                 break
#     except Exception:
#         pass

#     return out

# def _ensure_id3(file_mp3):
#     """Make sure file has an ID3 container."""
#     try:
#         ID3(str(file_mp3))
#     except ID3NoHeaderError:
#         audio = ID3()
#         audio.save(str(file_mp3))

# def _generate_cover_img(genre_str, size=1000):
#     """
#     Create a square PNG bytes with big initial(s) for the batch genre.
#     Examples: 'Salsa'→'S', 'Deep House'→'DH'
#     """
#     g = (_nz(genre_str).strip() or "GENRE").upper()
#     tokens = [t for t in re.split(r"\s+", g) if t]
#     initials = (tokens[0][0] + (tokens[1][0] if len(tokens) > 1 else ""))[:3]

#     img = Image.new("RGB", (size, size), (12, 12, 12))
#     draw = ImageDraw.Draw(img)
#     # simple vertical gradient
#     for i in range(size):
#         val = int(24 + (i / size) * 216)
#         draw.line([(0, i), (size, i)], fill=(val // 2, val, val // 3))

#     # pick a font robustly
#     font = None
#     for fname in ("DejaVuSans-Bold.ttf", "Arial.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"):
#         try:
#             font = ImageFont.truetype(fname, size=int(size * 0.55))
#             break
#         except Exception:
#             continue
#     if font is None:
#         font = ImageFont.load_default()

#     # center the initials
#     bbox = draw.textbbox((0, 0), initials, font=font)
#     tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
#     x = (size - tw) // 2
#     y = (size - th) // 2
#     # shadow + text
#     draw.text((x + 6, y + 6), initials, fill=(0, 0, 0), font=font)
#     draw.text((x, y), initials, fill=(255, 255, 255), font=font)

#     bio = io.BytesIO()
#     img.save(bio, format="PNG")
#     return bio.getvalue()

# def _embed_cover_mp3(mp3_path, png_bytes):
#     _ensure_id3(mp3_path)
#     audio = ID3(str(mp3_path))
#     # remove previous APIC frames
#     for k in list(audio.keys()):
#         if k.startswith("APIC"):
#             del audio[k]
#     # encoding=3 → UTF-8 (avoid importing Encoding to prevent version issues)
#     audio.add(APIC(encoding=3, mime="image/png", type=3, desc="Cover", data=png_bytes))
#     audio.save(v2_version=3)

# def _write_id3_to_aiff(aiff_path, tags, png_bytes):
#     """
#     Write ID3 tags + cover to AIFF using mutagen.aiff (ID3 in AIFF container).
#     """
#     af = AIFF(str(aiff_path))
#     id3 = af.tags if af.tags is not None else ID3()

#     # Core text frames (encoding=3 => UTF-8)
#     if tags.get("track_title"):  id3.add(TIT2(encoding=3, text=tags["track_title"]))
#     if tags.get("track_artist"): id3.add(TPE1(encoding=3, text=tags["track_artist"]))
#     if tags.get("mix_name"):     id3.add(TXXX(encoding=3, desc="MIXNAME", text=tags["mix_name"]))
#     if tags.get("key"):          id3.add(TKEY(encoding=3, text=tags["key"]))
#     if tags.get("bpm"):          id3.add(TBPM(encoding=3, text=tags["bpm"]))
#     if tags.get("genre"):        id3.add(TCON(encoding=3, text=tags["genre"]))
#     if tags.get("remixers"):     id3.add(TPE4(encoding=3, text=tags["remixers"]))
#     if tags.get("label"):        id3.add(TPUB(encoding=3, text=tags["label"]))

#     rel_iso = "-".join([_nz(tags.get("release_year")), _nz(tags.get("release_month")), _nz(tags.get("release_day"))]).strip("-")
#     if rel_iso: id3.add(TDRC(encoding=3, text=rel_iso))
#     pur_iso = "-".join([_nz(tags.get("purchase_year")), _nz(tags.get("purchase_month")), _nz(tags.get("purchase_day"))]).strip("-")
#     if pur_iso: id3.add(TXXX(encoding=3, desc="PurchaseDate", text=pur_iso))

#     if png_bytes:
#         for k in list(id3.keys()):
#             if k.startswith("APIC"):
#                 del id3[k]
#         id3.add(APIC(encoding=3, mime="image/png", type=3, desc="Cover", data=png_bytes))

#     af.tags = id3
#     af.save()

# def _ffmpeg_two_pass_loudnorm_to_aiff(inp, out_aiff, itgt=-14.0, lra=11.0, tp=-1.0):
#     """
#     Two-pass EBU R128 loudnorm (true-peak ceiling) to AIFF 16-bit/44.1k.
#     If pass-1 parsing fails, fall back to single-pass loudnorm.
#     Requires ffmpeg in PATH.
#     """
#     # Pass 1: measure
#     cmd1 = [
#         "ffmpeg", "-y", "-i", str(inp),
#         "-filter:a", f"loudnorm=I={itgt}:LRA={lra}:TP={tp}:print_format=json",
#         "-f", "null", "-"
#     ]
#     p1 = subprocess.run(cmd1, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
#     m = re.search(r'^\s*\{\s*"input_i".*\}\s*$', p1.stderr, flags=re.M | re.S)

#     if not m:
#         # Fallback: still loudnorm, single pass with the same targets
#         cmd_fallback = [
#             "ffmpeg", "-y", "-i", str(inp),
#             "-filter:a", f"loudnorm=I={itgt}:LRA={lra}:TP={tp}",
#             "-ar", "44100", "-sample_fmt", "s16", "-c:a", "pcm_s16be",
#             str(out_aiff)
#         ]
#         subprocess.run(cmd_fallback, check=True)
#         return

#     j = json.loads(m.group(0))
#     # Pass 2: apply measured params
#     cmd2 = [
#         "ffmpeg", "-y", "-i", str(inp),
#         "-filter:a",
#         ("loudnorm=I={I}:LRA={LRA}:TP={TP}:measured_I={mi}:measured_LRA={mlra}:"
#          "measured_TP={mtp}:measured_thresh={mth}:offset={off}:linear=true:print_format=summary"
#         ).format(I=itgt, LRA=lra, TP=tp, mi=j["input_i"], mlra=j["input_lra"],
#                  mtp=j["input_tp"], mth=j["input_thresh"], off=j["target_offset"]),
#         "-ar", "44100", "-sample_fmt", "s16", "-c:a", "pcm_s16be",
#         str(out_aiff)
#     ]
#     subprocess.run(cmd2, check=True)

# def _build_new_name(row):
#     nz = _nz
#     name = (
#         "TRkw_" + nz(row.get("track_title")) +
#         "_ARkw_" + nz(row.get("track_artist")) +
#         "_MXkw_" + nz(row.get("mix_name")) +
#         "_KYkw_" + nz(row.get("key")) +
#         "_BPkw_" + nz(row.get("bpm")) +
#         "_GNkw_" + nz(row.get("genre")) +
#         "_RMkw_" + nz(row.get("remixers")) +
#         "_LBkw_" + nz(row.get("label")) +
#         "_RYkw_" + nz(row.get("release_year")) + "_" + nz(row.get("release_month")) + "_" + nz(row.get("release_day")) +
#         "_PYkw_" + nz(row.get("purchase_year")) + "_" + nz(row.get("purchase_month")) + "_" + nz(row.get("purchase_day"))
#     )
#     # sanitize forbidden filesystem chars
#     return _safe_name(name)

# # -----######-----###### MAIN IMPORTABLE -----######-----###### #
# def _mp3_1808_covnormaiff_GET_df_results(
#     inputs,
#     batch_genre,
#     df_col_path="Path",
#     delete_original=False
# ):
#     """
#     Input:
#       - inputs: str (folder path) OR pandas.DataFrame with column df_col_path containing .mp3 paths
#       - batch_genre: str, e.g., "Salsa", "House"
#       - df_col_path: column name when inputs is a DataFrame
#       - delete_original: if True, delete original MP3 after successful AIFF

#     Output:
#       - df with appended columns:
#         ['track_title','track_artist','mix_name','key','bpm','genre','remixers','label',
#          'release_year','release_month','release_day','purchase_year','purchase_month','purchase_day',
#          'new_name','Path_mp3_renamed','Path_aiff','status','error']
#     """
#     # Build file list
#     if isinstance(inputs, pd.DataFrame):
#         paths = [Path(p) for p in inputs[df_col_path].tolist()]
#         df = inputs.copy()
#     else:
#         folder = Path(inputs)
#         paths = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".mp3"]
#         df = pd.DataFrame({df_col_path: [str(p) for p in paths]})

#     # Prepare columns
#     needed_cols = [
#         "track_title","track_artist","mix_name","key","bpm","genre","remixers","label",
#         "release_year","release_month","release_day",
#         "purchase_year","purchase_month","purchase_day",
#         "new_name","Path_mp3_renamed","Path_aiff","status","error"
#     ]
#     for c in needed_cols:
#         if c not in df.columns:
#             df[c] = ""

#     # Generate batch cover once
#     cover_png = _generate_cover_img(batch_genre)

#     # TQM BAR
#     pbar = tqdm(total=len(paths), desc="TQM • MP3→Cover+Rename+Loudnorm→AIFF", unit="file")

#     for p in paths:
#         status = "ok"; err = ""
#         mp3_in = Path(p)
#         try:
#             # 1) read tags
#             tags = _read_id3_fields(mp3_in)
#             if not tags.get("genre"):
#                 tags["genre"] = _nz(batch_genre)

#             # 2) embed cover into MP3 (non-fatal on failure)
#             try:
#                 _embed_cover_mp3(mp3_in, cover_png)
#             except Exception as e:
#                 err += f"[cover-mp3:{e}] "

#             # 3) strict new name (exact pattern)
#             new_name = _build_new_name(tags)

#             # 4) rename MP3 (preserves file dates)
#             mp3_new = mp3_in.with_name(new_name + mp3_in.suffix.lower())
#             if mp3_new != mp3_in:
#                 mp3_in.rename(mp3_new)
#                 mp3_in = mp3_new

#             # 5) normalize → AIFF 16-bit/44.1k, true-peak ceiling
#             aiff_out = mp3_in.with_suffix(".aiff")
#             _ffmpeg_two_pass_loudnorm_to_aiff(mp3_in, aiff_out, itgt=-14.0, lra=11.0, tp=-1.0)

#             # 6) write tags + cover to AIFF
#             try:
#                 _write_id3_to_aiff(aiff_out, tags, cover_png)
#             except Exception as e:
#                 status = "warn"
#                 err += f"[tags-aiff:{e}] "

#             # 7) optionally delete original MP3
#             if delete_original:
#                 try:
#                     mp3_in.unlink()
#                 except Exception as e:
#                     status = "warn"
#                     err += f"[del-mp3:{e}] "

#             # update df row
#             df.loc[df[df_col_path] == str(p), [
#                 "track_title","track_artist","mix_name","key","bpm","genre","remixers","label",
#                 "release_year","release_month","release_day",
#                 "purchase_year","purchase_month","purchase_day",
#                 "new_name","Path_mp3_renamed","Path_aiff","status","error"
#             ]] = [
#                 tags["track_title"], tags["track_artist"], tags["mix_name"], tags["key"], tags["bpm"], tags["genre"], tags["remixers"], tags["label"],
#                 tags["release_year"], tags["release_month"], tags["release_day"],
#                 tags["purchase_year"], tags["purchase_month"], tags["purchase_day"],
#                 new_name, str(mp3_in), str(aiff_out), status, err
#             ]

#         except subprocess.CalledProcessError as e:
#             df.loc[df[df_col_path] == str(p), ["status","error"]] = ["fail", f"[ffmpeg:{e}]"]
#         except Exception as e:
#             df.loc[df[df_col_path] == str(p), ["status","error"]] = ["fail", f"[misc:{e}]"]
#         finally:
#             pbar.update(1)
#     pbar.close()

#     df["new_name"] = df["new_name"].astype(str)
#     return df


# better function try 

In [ ]:
# # -----######-----###### CORE IMPORTABLE FUNCTION (SoundCloud DF Downloader — Append + Rate Limit + Backoff + Log) -----######-----###### #
# import os, re, time, random, math
# from pathlib import Path
# from datetime import datetime
# from tqdm import tqdm
# import pandas as pd

# def _sc__ensure_ffmpeg():
#     from shutil import which
#     return which("ffmpeg") is not None

# def _sc__build_opts(out_dir, ext, prefer_bitrate, archive_path, cookies_path, max_concurrent_frags=2):
#     postproc = []
#     if ext.lower() in ("mp3", "wav", "aiff"):
#         postproc = [{
#             "key": "FFmpegExtractAudio",
#             "preferredcodec": ext.lower(),
#             "preferredquality": str(prefer_bitrate) if ext.lower()=="mp3" else "0",
#         }]
#     outtmpl = str(Path(out_dir) / "%(uploader)s - %(title)s.%(ext)s")  # clean names

#     opts = {
#         "outtmpl": outtmpl,
#         "noplaylist": False,
#         "quiet": True,
#         "no_warnings": True,
#         "ignoreerrors": False,
#         "retries": 5,
#         "continuedl": True,
#         "format": "bestaudio/best",
#         "concurrent_fragment_downloads": int(max(1, max_concurrent_frags)),
#         "addmetadata": True,
#         "prefer_ffmpeg": True,
#         "postprocessors": postproc,
#         "nooverwrites": True,                     # never overwrite on disk
#         "download_archive": str(archive_path),    # skip duplicates by ID
#     }
#     if cookies_path:
#         opts["cookiefile"] = str(Path(cookies_path).expanduser())
#     return opts

# def _sc__extract_saved_path(info):
#     if not info:
#         return None
#     if "requested_downloads" in info and info["requested_downloads"]:
#         p = info["requested_downloads"][0].get("filepath")
#         return Path(p) if p else None
#     if "filepath" in info:
#         return Path(info["filepath"])
#     if "entries" in info and info["entries"]:
#         e = info["entries"][0]
#         if e and "requested_downloads" in e and e["requested_downloads"]:
#             p = e["requested_downloads"][0].get("filepath")
#             return Path(p) if p else None
#         if e and "filepath" in e:
#             return Path(e["filepath"])
#     return None

# def _sc__download_one(url, out_dir, ext, prefer_bitrate, archive_path, cookies_path, max_concurrent_frags):
#     import yt_dlp
#     if ext.lower() in ("mp3","wav","aiff") and not _sc__ensure_ffmpeg():
#         raise RuntimeError("ffmpeg not found. Install with: brew install ffmpeg")

#     ydl_opts = _sc__build_opts(out_dir, ext, prefer_bitrate, archive_path, cookies_path, max_concurrent_frags)

#     res = {
#         "timestamp": datetime.now().isoformat(timespec="seconds"),
#         "src_playlist": url, "dl_title": None, "dl_uploader": None, "dl_duration_sec": None,
#         "dl_id": None, "dl_ext": None, "dl_requested_ext": ext.lower(),
#         "dl_filepath": None, "dl_filesize_bytes": None, "dl_error": None, "dl_status": None
#     }

#     try:
#         with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#             info = ydl.extract_info(url, download=True)
#             if not info:
#                 res["dl_status"] = "skipped"   # likely skipped by archive
#                 return res

#         res["dl_title"]        = info.get("title")
#         res["dl_uploader"]     = info.get("uploader")
#         res["dl_duration_sec"] = info.get("duration")
#         res["dl_id"]           = info.get("id")
#         res["dl_ext"]          = info.get("ext")
#         res["dl_status"]       = "downloaded"

#         saved = _sc__extract_saved_path(info)
#         if saved and saved.exists():
#             res["dl_filepath"]       = str(saved)
#             res["dl_filesize_bytes"] = saved.stat().st_size

#     except Exception as e:
#         res["dl_error"]  = str(e)
#         res["dl_status"] = "failed"

#     return res


# # -----######-----###### MAIN IMPORTABLE FUNCTION -----######-----###### #
# def _sc_1808_df_GET_audio_APPEND_safe(
#     df_tracks,
#     out_dir,                      # REQUIRED: absolute path you provide
#     ext="mp3",
#     prefer_bitrate=320,
#     cookies_path=None,
#     existing_log_csv=None,        # optional: path to persistent CSV log (will be created/appended)
#     max_per_minute=30,            # polite rate: max URLs/min (≈ 2 req/s)
#     daily_cap=None,               # optional hard cap of downloads per run/day
#     max_retries_each=3,           # exponential backoff tries per URL
#     max_concurrent_frags=2        # keep per-file fragment concurrency modest
# ):
#     """
#     Append-only SoundCloud downloader with rate limiting, backoff, and persistent CSV logging.
#     Input:
#       - df_tracks: must contain 'src_playlist' column with SoundCloud URLs.
#       - out_dir: folder where audio will be saved (never overwritten).
#     Behavior:
#       - Uses a download archive in out_dir to skip already downloaded IDs.
#       - Clean filenames (Artist - Title.ext).
#       - Appends rows to an existing CSV log if provided.
#       - Respects max_per_minute with random jitter; exponential backoff on failures.
#     Returns:
#       - df_log: a DataFrame of ONLY the rows from this run (you can read/concat the CSV for full history).
#     """
#     if "src_playlist" not in df_tracks.columns:
#         raise ValueError("df_tracks must contain 'src_playlist' with SoundCloud URLs.")

#     out_dir = Path(out_dir).expanduser().resolve()
#     out_dir.mkdir(parents=True, exist_ok=True)
#     archive_path = out_dir / "_download_archive.txt"
#     archive_path.touch(exist_ok=True)

#     # Load existing CSV log (for your reference). We still return just-this-run rows.
#     if existing_log_csv:
#         existing_log_csv = str(Path(existing_log_csv).expanduser())
#         if not Path(existing_log_csv).exists():
#             Path(existing_log_csv).parent.mkdir(parents=True, exist_ok=True)

#     urls = [u.strip() for u in df_tracks["src_playlist"].astype(str).fillna("")]
#     urls = [u for u in urls if u]  # drop blanks

#     # Rate limiting params
#     min_interval = 60.0 / max(1, max_per_minute)  # seconds per URL
#     last_ts = 0.0
#     successes_this_run = 0
#     emitted = []

#     for idx, url in enumerate(tqdm(urls, desc="TQM • Downloading SoundCloud audio", unit="url")):
#         # Daily cap stop
#         if daily_cap is not None and successes_this_run >= daily_cap:
#             emitted.append({
#                 "timestamp": datetime.now().isoformat(timespec="seconds"),
#                 "src_playlist": url,
#                 "dl_title": None, "dl_uploader": None, "dl_duration_sec": None,
#                 "dl_id": None, "dl_ext": None, "dl_requested_ext": ext.lower(),
#                 "dl_filepath": None, "dl_filesize_bytes": None,
#                 "dl_error": "daily_cap_reached", "dl_status": "skipped_cap"
#             })
#             continue

#         # Polite pacing with jitter
#         now = time.time()
#         wait_needed = (last_ts + min_interval) - now
#         if wait_needed > 0:
#             time.sleep(wait_needed)
#         # Add small random jitter to avoid looking robotic
#         time.sleep(random.uniform(0.05, 0.35))
#         last_ts = time.time()

#         # Retry with exponential backoff
#         attempt = 0
#         result = None
#         while attempt < max_retries_each:
#             result = _sc__download_one(
#                 url=url,
#                 out_dir=out_dir,
#                 ext=ext,
#                 prefer_bitrate=prefer_bitrate,
#                 archive_path=archive_path,
#                 cookies_path=cookies_path,
#                 max_concurrent_frags=max_concurrent_frags
#             )

#             # Consider "downloaded" or "skipped" by archive a success
#             if result.get("dl_status") in ("downloaded", "skipped"):
#                 break

#             # Backoff on failure (network/429/etc.)
#             attempt += 1
#             if attempt < max_retries_each:
#                 sleep_s = (2 ** attempt) + random.uniform(0, 0.6)
#                 time.sleep(sleep_s)

#         emitted.append(result)
#         if result.get("dl_status") in ("downloaded", "skipped"):
#             successes_this_run += 1

#         # Persist row-by-row to CSV to never lose progress
#         if existing_log_csv:
#             pd.DataFrame([result]).to_csv(existing_log_csv, mode="a", header=not Path(existing_log_csv).exists(), index=False)

#     df_run = pd.DataFrame(emitted)
#     return df_run


# # REQUIRED: your output folder
# OUT_DIR = "/Users/yerik/Downloads/_soundcloud_audio"

# # Optional persistent log (grows forever, append-only)
# LOG_CSV = "/Users/yerik/Downloads/_soundcloud_audio/_download_log.csv"

# # mp3/wav/aiff settings
# EXT   = "mp3"
# KBPS  = 320
# COOKS = None  # e.g. "/Users/yerik/Downloads/cookies.txt" if needed

# # Politeness knobs (tune if you see throttling)
# MAX_PER_MIN   = 30     # ≈2 req/s is usually safe; drop to 10 if you see 429s
# DAILY_CAP     = None   # e.g. 300 if you want a hard ceiling
# RETRIES_EACH  = 3
# FRAG_CONCUR   = 2

# df_log_run = _sc_1808_df_GET_audio_APPEND_safe(
#     df_tracks,
#     out_dir=OUT_DIR,
#     ext=EXT,
#     prefer_bitrate=KBPS,
#     cookies_path=COOKS,
#     existing_log_csv=LOG_CSV,
#     max_per_minute=MAX_PER_MIN,
#     daily_cap=DAILY_CAP,
#     max_retries_each=RETRIES_EACH,
#     max_concurrent_frags=FRAG_CONCUR
# )

# # Tip: if you want the full history in memory:
# # import pandas as pd
# # df_full = pd.read_csv(LOG_CSV)
